# Session 16 · Case study: drift, retraining and the final leaderboard round (L4)

**Block 3 practice.** The lecturer has released the labels of the 2024 decisions as feedback data (`case-study/data/feedback_2024.csv`, 40,369 decisions). You

1. check the 2024–2026 test decisions for **input drift** before any label is known (description length, language, issuing country, out-of-vocabulary words) and for **prediction drift**;
2. detect the **label shift** once the 2024 labels arrive (chapter and heading shares, headings never seen in training) and measure the accuracy of version 1 on 2024;
3. compare **retraining candidates** honestly on the last quarter of 2024;
4. retrain version 2 with the 2024 labels and make the **final leaderboard submission (L4)**, ranked on the hidden 2025–2026 decisions;
5. collect the numbers for the model card.

The model is the word TF-IDF + linear SVM of Session 13, trained on the 50,000-decision sample (about 10 seconds per fit). The optional last section repeats the comparison on the full training set.

**Feedback file.** The notebook looks for `case-study/data/feedback_2024.csv` (or the path in the environment variable `FEEDBACK_PATH`). The lecturer creates it with `make_feedback_2024.py`; without it, sections 3–5 are skipped.

Theory: [03-monitoring-and-maintenance.md](../theory/03-monitoring-and-maintenance.md). The same statistics, as tested functions, are in `../workspace/src/tariff_service/drift.py`.

*Author: course team, licence CC-BY-4.0.*

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
sys.path.insert(0, str(ROOT / "sessions" / "16-deployment-and-monitoring" / "workspace" / "src"))
from tariff_service.drift import ks_test, psi, psi_categories, psi_from_shares, unseen_share  # noqa: E402

train = pd.read_parquet(DATA / "train_sample.parquet")
test = pd.read_parquet(DATA / "test.parquet")
nomenclature = pd.read_parquet(DATA / "nomenclature.parquet").set_index("heading")
test["period"] = np.where(test["start_date"].dt.year == 2024, "2024", "2025-26")
print(len(train), "training decisions 2017-2023 |", test["period"].value_counts().to_dict())

FEEDBACK = Path(os.environ.get("FEEDBACK_PATH", DATA / "feedback_2024.csv"))
feedback = pd.read_csv(FEEDBACK, dtype=str) if FEEDBACK.exists() else None
print("feedback:", "not released yet" if feedback is None else f"{len(feedback):,} labelled decisions of 2024")

## 1. Version 1: the model in service

In [ ]:
def make_model(alpha=1e-5):
    return make_pipeline(TfidfVectorizer(min_df=2, sublinear_tf=True, max_features=400_000),
                         SGDClassifier(loss="hinge", alpha=alpha, max_iter=20, tol=None, random_state=0, n_jobs=-1))


v1 = make_model().fit(train["description"], train["heading"])
test["pred_v1"] = v1.predict(test["description"])
vocab = set(v1.named_steps["tfidfvectorizer"].vocabulary_)
print("headings known to v1:", len(v1.classes_))

## 2. Input drift and prediction drift (no labels needed)

Compare each test period with the training data: description length (KS statistic and PSI), language and issuing-country shares (PSI), the out-of-vocabulary rate, and the shares of the **predicted** chapters.

In [ ]:
TOKEN = re.compile(r"(?u)\b\w\w+\b")


def oov_rate(texts):
    words = [w for t in texts for w in TOKEN.findall(t.lower())]
    return sum(w not in vocab for w in words) / len(words)


def shares(values):
    return pd.Series(values).value_counts(normalize=True).to_dict()


log_len_train = np.log1p(train["description"].str.len())
recent_train = train[train["start_date"].dt.year == 2023]
rows = [{"period": "2023 (inside training)", "n": len(recent_train),
         "KS_log_length": ks_test(log_len_train, np.log1p(recent_train["description"].str.len())).statistic,
         "PSI_log_length": psi(log_len_train, np.log1p(recent_train["description"].str.len())),
         "PSI_language": psi_categories(shares(train["language"]), recent_train["language"]),
         "PSI_country": psi_categories(shares(train["issuing_country"]), recent_train["issuing_country"]),
         "OOV_rate": oov_rate(recent_train["description"].sample(5000, random_state=0)),
         "PSI_predicted_chapter": psi_categories(shares(train["chapter"]),
                                                 pd.Series(v1.predict(recent_train["description"])).str[:2])}]
for period, part in test.groupby("period"):
    lengths = np.log1p(part["description"].str.len())
    rows.append({"period": period, "n": len(part),
                 "KS_log_length": ks_test(log_len_train, lengths).statistic,
                 "PSI_log_length": psi(log_len_train, lengths),
                 "PSI_language": psi_categories(shares(train["language"]), part["language"]),
                 "PSI_country": psi_categories(shares(train["issuing_country"]), part["issuing_country"]),
                 "OOV_rate": oov_rate(part["description"].sample(5000, random_state=0)),
                 "PSI_predicted_chapter": psi_categories(shares(train["chapter"]), part["pred_v1"].str[:2])})
input_drift = pd.DataFrame(rows).set_index("period")
# the 2023 row is part of the training data: its values show the level of "no drift" (words seen only once
# are not in the vocabulary because of min_df=2; GB was already absent in 2023)
input_drift.round(3)

In [ ]:
country = pd.DataFrame({"train 2017-2023": train["issuing_country"].value_counts(normalize=True),
                        "test 2024": test.loc[test["period"] == "2024", "issuing_country"].value_counts(normalize=True),
                        "test 2025-26": test.loc[test["period"] == "2025-26", "issuing_country"].value_counts(normalize=True)})
print(pd.crosstab(train["start_date"].dt.year, train["issuing_country"] == "GB").rename(columns={True: "GB"})["GB"].to_dict())
country.fillna(0).sort_values("train 2017-2023", ascending=False).head(8).round(3)

**Reading the result.** The issuing countries shift most: the United Kingdom issued about 4 % of the training decisions (all of them in English) and none after Brexit (2021); the English share of the descriptions falls from about 5 % to 1 %. Description lengths and the predicted chapters move only a little. The out-of-vocabulary rate rises with distance from the training period: new product names, new technical terms.

## 3. The 2024 labels arrive: label shift and the accuracy of version 1

In [ ]:
if feedback is not None:
    fb = test.merge(feedback, on="id")
    y2023 = train[train["start_date"].dt.year == 2023]
    chapters = pd.DataFrame({"2023": y2023["chapter"].value_counts(normalize=True),
                             "2024": fb["heading"].str[:2].value_counts(normalize=True)}).fillna(0)
    chapters["change"] = chapters["2024"] - chapters["2023"]
    print("PSI of chapter shares 2023 -> 2024:", round(psi_from_shares(chapters["2023"], chapters["2024"]), 3))
    headings_psi = psi_categories(shares(y2023["heading"]), fb["heading"])
    print("PSI of heading shares 2023 -> 2024:", round(headings_psi, 3))
    print("2024 decisions with a heading v1 never saw:", round(unseen_share(v1.classes_, fb["heading"]), 4))
    acc_v1 = accuracy_score(fb["heading"], fb["pred_v1"])
    print(f"v1 on 2024: accuracy {acc_v1:.3f}, macro-F1 {f1_score(fb['heading'], fb['pred_v1'], average='macro'):.3f}")
    display(chapters.reindex(chapters["change"].abs().sort_values(ascending=False).index).head(8).round(4))
else:
    print("No feedback file yet: skipped.")

In [ ]:
if feedback is not None:
    unseen = fb.loc[~fb["heading"].isin(v1.classes_), "heading"].value_counts().head(8)
    print(pd.DataFrame({"n": unseen, "text": nomenclature["heading_description"].reindex(unseen.index).str[:60]}))
    by_lang = (fb.assign(correct=fb["heading"] == fb["pred_v1"])
               .groupby("language").agg(n=("correct", "size"), accuracy=("correct", "mean")))
    display(by_lang.sort_values("n", ascending=False).head(8).round(3))

**Accuracy by language.** The differences are large: in the course team's run, about 0.96 for German and 0.37–0.64 for the other main languages. Two facts explain most of it: German makes up 57 % of the training data, and 44 % of the German 2024 descriptions quote their own heading number (Session 13), against almost none in the other languages. A model card must report such subgroups; the overall accuracy hides them.

**Reading the result.** The chapter mix shifts moderately (chapter 85 falls by about three points, 84 and 64 rise), and a few 2024 headings were never seen in training. Yet the accuracy of v1 on 2024 is *higher* than its validation accuracy of about 0.77 on 2022–2023 (Session 13): the final v1 was trained on 2017–2023, the years closest to 2024. A retraining rule that waits for a drop in accuracy would not fire. The next section asks whether retraining helps anyway.

## 4. Retraining candidates, compared honestly

The 2024 labels are both new training data and the only data close to the target period. To compare candidates without touching the 2025–2026 labels, hold out the last quarter of 2024 (October–December) and train on the rest.

In [ ]:
if feedback is not None:
    late = fb["start_date"] >= "2024-10-01"
    fb_early, fb_late = fb[~late], fb[late]
    cols = ["description", "heading", "start_date"]
    candidates = {
        "A: v1 as it is (2017-2023)": train[cols],
        "B: + 2024 Jan-Sep": pd.concat([train[cols], fb_early[cols]]),
        "C: window 2020-2023 + 2024 Jan-Sep (forget old years)":
            pd.concat([train.loc[train["start_date"].dt.year >= 2020, cols], fb_early[cols]]),
    }
    rows = []
    for name, data in candidates.items():
        m = make_model().fit(data["description"], data["heading"])
        p = m.predict(fb_late["description"])
        rows.append({"candidate": name, "n_train": len(data),
                     "accuracy Oct-Dec 2024": accuracy_score(fb_late["heading"], p),
                     "macro-F1 Oct-Dec 2024": f1_score(fb_late["heading"], p, average="macro")})
    display(pd.DataFrame(rows).set_index("candidate").round(3))

**Question.** Is the difference between B and C larger than the noise? With about 10,000 validation decisions, the standard error of an accuracy near 0.8 is about 0.004 (Session 7). Which candidate would you release, and why?

## 5. Version 2 and the final submission (L4)

In [ ]:
if feedback is not None:
    v2_data = pd.concat([train[["description", "heading"]], fb[["description", "heading"]]])
    v2 = make_model().fit(v2_data["description"], v2_data["heading"])
    private = test[test["period"] == "2025-26"]
    submission = pd.DataFrame({"id": test["id"], "heading": v2.predict(test["description"])})
    assert len(submission) == len(test) and submission["heading"].str.fullmatch(r"\d{4}").all()
    submission.to_csv("submission_L4.csv", index=False)
    print("written: submission_L4.csv |", len(v2_data), "training decisions")

Upload `submission_L4.csv` to the leaderboard, together with a one-page description of your method. Only the private part (2025–2026) counts for the final ranking; on the public part (2024) version 2 has seen the labels and scores near 1.0, which says nothing. The next cell scores v1 and v2 locally if the lecturer's solution file is available.

In [ ]:
solution_file = DATA / "instructor" / "solution.csv"
if feedback is not None and solution_file.exists():
    solution = pd.read_csv(solution_file, dtype=str)
    priv = test.merge(solution[solution["Usage"] == "Private"], on="id")
    p2 = v2.predict(priv["description"])
    for name, pred in [("v1 (2017-2023)", priv["pred_v1"]), ("v2 (+ 2024 feedback)", p2)]:
        print(f"{name:22s} private 2025-26: accuracy {accuracy_score(priv['heading'], pred):.3f}, "
              f"macro-F1 {f1_score(priv['heading'], pred, average='macro'):.3f}")
else:
    print("Solution file not available: upload the submission instead.")

## 6. Optional: the full training set

Set `RUN_FULL = True` to repeat v1 and v2 with all 309,529 training decisions (about 4 minutes per fit and 8 GB of memory). Reference values from the course team's run: private accuracy 0.847 (v1) and 0.857 (v2).

In [ ]:
RUN_FULL = False
if RUN_FULL and feedback is not None:
    full = pd.read_parquet(DATA / "train.parquet", columns=["description", "heading"])
    big_v1 = make_model(2e-6).fit(full["description"], full["heading"])
    big_v2 = make_model(2e-6).fit(pd.concat([full, fb[["description", "heading"]]])["description"],
                                  pd.concat([full, fb[["description", "heading"]]])["heading"])
    pd.DataFrame({"id": test["id"], "heading": big_v2.predict(test["description"])}).to_csv(
        "submission_L4_full.csv", index=False)
    print("written: submission_L4_full.csv")

## 7. Numbers for the model card

Collect the values for `../workspace/MODEL_CARD.md` (version 2): training data and period, validation scheme and scores, the drift observed (country, language, chapter shares, unseen headings), the retraining decision and its evidence, and the limitations you found.

## Your turn

1. Write the monitoring section of the model card: which signals fired, which did not, and what the retraining comparison showed.
2. A colleague proposes to retrain only when accuracy on new labels drops by three points. Using the results above, argue for or against this rule.
3. Which drift signal would you watch for the next HS revision (2027), and how would you detect headings that the model cannot predict at all?
4. Optional: repeat section 4 with the character n-gram model of Session 13. Does the better model change the retraining decision?

In [ ]:
# your work here